In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from utils import trainer as tr
from torch.utils.data import Dataset
import random
import torch.backends.cudnn as cudnn
import numpy as np
from trained_models.MNIST.model4 import trainer as t1
from trained_models.MNIST.model5 import trainer as t2
from trained_models.MNIST.model6 import trainer as t3
import numpy as np
from sklearn.model_selection import train_test_split
from torch.linalg import norm
from torchvision import models
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
from copy import deepcopy
from torch.nn.functional import pad
from torch.func import jacrev
from visdom import Visdom

In [2]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
#device='cpu'
print(f"Using device: {device}")

Using device: cuda:0


In [3]:
!nvidia-sml

/usr/bin/sh: 1: nvidia-sml: not found


In [4]:
torch.cuda.empty_cache()

In [21]:
trainloader1, valloader1, testloader1, init_net1, net1 = t1.train_net(force_train=False)

Model weights loaded successfully.


In [14]:
trainloader2, valloader2, testloader2, init_net2, net2 = t2.train_net(force_train=False)   

Model weights loaded successfully.


In [3]:
trainloader3, valloader3, testloader3, init_net3, net3 = t3.train_net(force_train=True)

Model weights loaded successfully.
NUMBER OF PARAMS:  211220


/work/DLR/trained_models/MNIST/model6/model6.py:39: UserWarning: Implicit dimension choice for log_softmax has been deprecated. Change the call to include dim=X as an argument.
  return F.log_softmax(x)


Time:  6.756708145141602
Epoch:  0 Train Loss:  0.10005013 Test Loss:  0.08744699 Train Acc:  98.43125 Test Acc:  97.34 Best Val Acc:  97.525 Best Val Loss:  0.075711995 Best Test Acc:  97.34 Best Test Loss:  0.08744699
Time:  6.159066438674927
Epoch:  1 Train Loss:  0.052008793 Test Loss:  0.08334088 Train Acc:  98.925 Test Acc:  97.21 Best Val Acc:  97.70833333333333 Best Val Loss:  0.07480353 Best Test Acc:  97.21 Best Test Loss:  0.08334088
Time:  6.227961778640747
Epoch:  2 Train Loss:  0.038083427 Test Loss:  0.11393126 Train Acc:  98.8 Test Acc:  96.91 Best Val Acc:  97.70833333333333 Best Val Loss:  0.07480353 Best Test Acc:  97.21 Best Test Loss:  0.08334088
Time:  6.347402811050415
Epoch:  3 Train Loss:  0.02815133 Test Loss:  0.097416475 Train Acc:  99.41458333333334 Test Acc:  97.44 Best Val Acc:  97.84166666666667 Best Val Loss:  0.07480353 Best Test Acc:  97.44 Best Test Loss:  0.08334088
Time:  6.355436563491821
Epoch:  4 Train Loss:  0.025475845 Test Loss:  0.0984451 Tr

In [8]:
#GCNN
print(tr.get_acc_ce(net1.to(device), testloader1))

99.24


In [9]:
#CNN without augmentation
print(tr.get_acc_ce(net2.to(device), testloader2))

98.97


In [4]:
#CNN with random rotation augmentation
print(tr.get_acc_ce(net3.to(device), testloader3))

97.71


In [15]:
vis = Visdom()
def is_rotated_version(filter1, filter2, tolerance=1e-5):
        
    # Detach tensors and move to CPU for consistent operations
    filter1 = filter1.detach().cpu()
    filter2 = filter2.detach().cpu()

    for k in range(4): # k=0,1,2,3 for 0, 90, 180, 270 degrees
        # Rotate the last two dimensions (height and width)
        rotated_filter1 = torch.rot90(filter1, k=k, dims=[0,1])
        
        # Compare the rotated filter to the second filter.
        # Use torch.allclose for robust floating-point comparison.
        if torch.allclose(rotated_filter1, filter2, atol=tolerance):
            return True
    
    return False
 

def group_rotated_filters(filters, tolerance=1e-5):
    groups = []
    # Create a list of booleans to track which filters have been grouped
    is_grouped = [False] * len(filters)

    for i in range(len(filters)):
        if not is_grouped[i]:
            # Start a new group with the current filter
            filter1 = filters[i].mean(dim=0)
            current_group = [filter1]
            is_grouped[i] = True
            
            # Compare this filter to all subsequent filters
            for j in range(i + 1, len(filters)):
                if not is_grouped[j]:
                    filter2= filters[j].mean(dim=0)
                    # If it's a rotated version, add it to the current group
                    if is_rotated_version(filter1, filter2, tolerance):
                        current_group.append(filter2)
                        is_grouped[j] = True
            
            groups.append(current_group)
            
    
    return groups


def visualize_filter_groups(groups):

    print(f"Found {len(groups)} unique filter groups. Visualizing each group in a separate window.")
    
    for i, group in enumerate(groups):
        # Check if the group is empty
        if not group:
            print(f"Skipping visualization for group {i + 1}: Group is empty.")
            continue
            
        vis_data = []
        filter_averages = []
        for filter_tensor in group:
            
            # Ensure the tensor is on CPU and detached
            filter_tensor = filter_tensor.detach().cpu()
            
            # If it's already a 2D tensor [H, W], just add a channel dimension
            reshaped_filter = filter_tensor.unsqueeze(0)
                
            # Add a batch dimension to make it compatible with visdom.images
            #reshaped_filter = reshaped_filter.unsqueeze(0)
            
            vis_data.append(reshaped_filter)

        title = f"Filter Group {i + 1} (Size: {len(group)})"
        # Display the group using vis.images
        vis.images(
            vis_data,
            win=f'filter_group_{i}', # Unique window ID for each group
            # Since we now have one image per filter, the number of rows is just the number of filters
            nrow=len(group), 
            opts=dict(
                title=title,
                caption=f'Filters in Group {i + 1}',
                fillcolor='rgba(0,0,0,0)',
                padding=5,
            )
        )


Setting up a new session...


In [12]:
g= group_rotated_filters(net2.features[2].weight, tolerance= 0.015)
print(len(g))
print(g[0][0].shape)

NameError: name 'net2' is not defined

In [21]:
g= group_rotated_filters(net3.features[2].weight, tolerance= 0.01)
print(len(g))

26


In [24]:
visualize_filter_groups(g)

Found 26 unique filter groups. Visualizing each group in a separate window.
